# 07 · Caps, Floors, and Swaptions

Rate option pricers sit on top of the same curve data as swaps. Supply discount/forward curves plus a volatility surface and request Greeks alongside PV.

### Learning Objectives
- Build a reusable interest-rate options market (discount + forward + cap/swaption vols)
- Price caps and floors with delta/vega metrics
- Create a payer swaption with expiry/tenor schedules and inspect its Greeks

In [ ]:
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, ForwardCurve
from finstack.core.market_data.surfaces import VolSurface
from finstack.valuations.instruments import InterestRateOption, Swaption
from finstack.valuations.pricer import create_standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9975), (1.0, 0.9940), (3.0, 0.9700), (5.0, 0.9350), (7.0, 0.8950)],
    )
)
market.insert_forward(
    ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.0350), (1.0, 0.0360), (3.0, 0.0385), (5.0, 0.0400), (7.0, 0.0410)],
        base_date=as_of,
    )
)
market.insert_surface(
    VolSurface(
        "USD-CAP-VOL",
        expiries=[0.5, 1.0, 2.0, 5.0],
        strikes=[0.01, 0.02, 0.03, 0.04],
        grid=[
            [0.38, 0.36, 0.34, 0.32],
            [0.35, 0.33, 0.31, 0.30],
            [0.32, 0.30, 0.29, 0.28],
            [0.28, 0.27, 0.26, 0.25],
        ],
    )
)
market.insert_surface(
    VolSurface(
        "USD-SWAPTION-VOL",
        expiries=[1.0, 2.0, 5.0, 7.0],
        strikes=[0.02, 0.03, 0.04],
        grid=[
            [0.30, 0.29, 0.28],
            [0.28, 0.27, 0.26],
            [0.26, 0.25, 0.24],
            [0.25, 0.24, 0.23],
        ],
    )
)

registry = create_standard_registry()
notional = Money(10_000_000, USD)


## 1. Caps & Floors
`InterestRateOption.cap/floor` constructors only need notionals, strike, start/end, and references to discount/forward/vol IDs. Request delta/vega alongside PV.

In [ ]:
cap = InterestRateOption.cap(
    "USD-CAP-5Y",
    notional,
    strike=0.04,
    start_date=as_of + timedelta(days=2),
    end_date=date(2029, 1, 2),
    discount_curve="USD-OIS",
    forward_curve="USD-SOFR-3M",
    vol_surface="USD-CAP-VOL",
)
cap_res = registry.price_with_metrics(cap, "discounting", market, ["delta", "vega"])
print("Cap PV:", cap_res.value)
print("Cap delta:", cap_res.measures.get("delta"))
print("Cap vega:", cap_res.measures.get("vega"))

floor = InterestRateOption.floor(
    "USD-FLOOR-5Y",
    notional,
    strike=0.02,
    start_date=as_of + timedelta(days=2),
    end_date=date(2029, 1, 2),
    discount_curve="USD-OIS",
    forward_curve="USD-SOFR-3M",
    vol_surface="USD-CAP-VOL",
)
floor_pv = registry.price(floor, "discounting", market)
print("Floor PV:", floor_pv.value)


## 2. Payer Swaption
Swaptions wrap an underlying swap (expiry × tenor). Provide expiry, swap start/end, strike, and the volatility surface ID.

In [ ]:
swaption = Swaption.payer(
    "USD-SWAPTION-1Yx5Y",
    notional,
    strike=0.0325,
    expiry=date(2025, 1, 2),
    swap_start=date(2025, 1, 2),
    swap_end=date(2030, 1, 2),
    discount_curve="USD-OIS",
    forward_curve="USD-SOFR-3M",
    vol_surface="USD-SWAPTION-VOL",
    exercise="european",
    settlement="physical",
)
swaption_res = registry.price_with_metrics(swaption, "discounting", market, ["delta", "vega"])
print("Swaption PV:", swaption_res.value)
print("Swaption delta:", swaption_res.measures.get("delta"))
print("Swaption vega:", swaption_res.measures.get("vega"))


## Summary
- Discount/forward curves shared with swaps also serve the rate option engines.
- Volatility surfaces are simple expiry/strike grids keyed by ID.
- Caps/floors and swaptions expose the usual Greeks, letting you hedge exposures with DV01 or option vega ladders.